<a href="https://colab.research.google.com/github/adryann-olivatti/IC_cozinhas_solidarias/blob/main/documentacao_metodologia_cozinhas_solidarias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Documentação e Guia Metodológico: Análise de Renda das Organizações Fornecedoras

Este notebook serve como **guia explicativo e metodológico** para a análise de dados do **Programa de Aquisição de Alimentos (PAA) — Cozinhas Solidárias (2025)**.

Ele detalha de onde os dados foram extraídos, o papel de cada fonte de informação, o dicionário de variáveis e como funciona a lógica do código de análise.

## 1. De onde vieram os dados? (Origem e Linhagem)

A análise utiliza 3 fontes principais interligadas:

### A. Planilha de Dados Operacionais da CONAB (Google Sheets)
- **Link:** [Dados CONAB PAA Cozinhas Solidárias 2025](https://docs.google.com/spreadsheets/d/1O_mmcnZ2QCBkA6juq_TzKYFv6KDwHsCU/edit)
- **O que é:** É a **base de dados bruta e operacional** mantida pela Companhia Nacional de Abastecimento (CONAB) em parceria com o Ministério do Desenvolvimento e Assistência Social, Família e Combate à Fome (MDS).
- **Conteúdo:** Registra individualmente cada item de alimento fornecido, incluindo o valor financeiro em Reais (R$), a quantidade em kg, a organização fornecedora responsável (Proponente) e o público prioritário.

### B. Catálogo de Dados em JSON (GitHub)
- **Link:** [GitHub - dataset_catalogue.json](https://github.com/TriangulosTecnologia/cozsolidarias/blob/main/public/dataset_catalogue.json)
- **O que é:** O **esquema estruturado de metadados** da plataforma Cozinhas Solidárias em Rede.
- **Conteúdo:** Indexa os conjuntos de dados do projeto por ID, registrando a proveniência, licenças, versão da estrutura e relacionamentos entre datasets.

### C. Portal de Metadados e Vocabulário
- **Link:** [Portal de Dados - Cozinhas Solidárias](https://cozsolidarias.triangulos.tech/dados)
- **O que é:** A interface pública de documentação do projeto.
- **Conteúdo:** Define as regras de interpretação dos dados (histórico acumulado, exaustividade, granularidade e convenções territoriais).

## 2. Dicionário dos Dados da CONAB

Na planilha da CONAB, cada linha representa **um item de alimento fornecido**. As colunas chave utilizadas na análise são:

| Coluna Original | Nome Tratado | Descrição | Exemplo |
| :--- | :--- | :--- | :--- |
| `NOME PROPONENTE` | `organizacao` | Nome da entidade/organização fornecedora | COOPERATIVA DE PRODUTORES... |
| `CNPJ PROPONENTE` | `cnpj` | CNPJ único da organização | 09.197.432/0001-09 |
| `TIPO DE ORG` | `tipo_org` | Categoria/Público prioritário declarado | AGRICULTOR FAMILIAR, QUILOMBOLA |
| `VALOR (R$)` | `valor` | Valor financeiro previsto em Reais | R$ 15,000.00 |
| `PRODUTO` | `produto` | Item agrícola ou processado fornecido | FEIJAO, BANANA, OVOS |
| `UF CONAB` | `uf` | Estado de origem da organização | AC, AL, AM, AP |

## 3. Como Funciona a Lógica da Análise (Passo a Passo)

O notebook principal de análise executa o pipeline em 5 etapas:

```
+-----------------------------+
| 1. Download & Busca de Abas |
+-----------------------------+
              |
              v
+-----------------------------+
| 2. Limpeza Monetária (Float)|
+-----------------------------+
              |
              v
+-----------------------------+
| 3. Agrupamento por CNPJ     |
+-----------------------------+
              |
              v
+-----------------------------+
| 4. Consolidação por Categoria|
+-----------------------------+
              |
              v
+-----------------------------+
| 5. Visualização & Exportação|
+-----------------------------+
```

### Detalhamento dos Passos:

1. **Download Dinâmico das Abas:** Como a planilha do Google Sheets possui múltiplas abas e linhas de cabeçalho institucional antes da tabela, o código inspeciona todas as abas e localiza automaticamente onde está a palavra-chave `PROPONENTE` ou `CNPJ`.
2. **Limpeza Financeira:** O campo de valor vem formatado como texto (ex: `"R$ 15,000.00"`). A função `limpar_valor()` remove caracteres não numéricos e converte o texto para o tipo numérico decimal (`float`).
3. **Agrupamento por Organização (CNPJ):** Como uma organização fornece múltiplos alimentos, o código soma todos os valores atribuídos a um mesmo CNPJ para calcular a **Renda Total por Organização**.
4. **Agrupamento por Tipo:** Os totais são reagrupados por `TIPO DE ORG` para responder à pergunta central: qual a renda recebida por categoria de produtores (ex.: assentados, agricultores familiares, quilombolas, pescadores).
5. **Visualização e Saída:** Geração de gráficos de barras horizontais e exportação de relatórios em formato CSV/Excel.

## 4. Código Demonstrativo: Inspecionando o Catálogo de Dados (JSON)

In [ ]:
import requests
import json

# Consulta dinâmica dos metadados do repositório no GitHub
url_catalogo = "https://raw.githubusercontent.com/TriangulosTecnologia/cozsolidarias/main/public/dataset_catalogue.json"

try:
    response = requests.get(url_catalogo)
    if response.status_code == 200:
        catalogo = response.json()
        print("--- METADADOS DO CATÁLOGO DE DADOS COZINHAS SOLIDÁRIAS ---")
        print(f"Versão do Esquema: {catalogo.get('schema_version', 'N/A')}")
        print(f"Data de Atualização: {catalogo.get('updated_at', 'N/A')}")
        print(f"Total de Coleções Registradas: {len(catalogo.get('collections', []))}")
    else:
        print("Não foi possível acessar o catálogo em JSON remotamente.")
except Exception as e:
    print(f"Erro ao consultar catálogo: {e}")